In [ ]:
import random
import time
from IPython.display import clear_output

PEAS = {
    "P - Performance": "Hút sạch bụi, tránh vật cản, số bước càng ít càng tốt",
    "E - Environment": "Phòng 3x3, có ô sạch, ô bẩn và có vật cản",
    "A - Actuators": "SUCK, UP, DOWN, LEFT, RIGHT, STOP",
    "S - Sensors": "Biết vị trí hiện tại, biết ô hiện tại có bụi không, biết action hợp lệ, biết vật cản xung quanh"
}

ROWS = 3
COLS = 3

ACTIONS = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}

SCAN_PATH = [
    (0, 0), (0, 1), (0, 2),
    (1, 2), (1, 1), (1, 0),
    (2, 0), (2, 1), (2, 2)
]


def create_environment(agent_pos, dirt_cells, obstacle_cells):
    return {
        "agent_pos": agent_pos,
        "dirt": set(dirt_cells),
        "obstacles": set(obstacle_cells),
        "initial_dirt_count": len(dirt_cells),
        "step": 0,
        "last_action": None
    }


def is_inside(row, col):
    return 0 <= row < ROWS and 0 <= col < COLS


def get_neighbor_info(env):
    row, col = env["agent_pos"]
    info = {}

    for action, (dr, dc) in ACTIONS.items():
        new_row = row + dr
        new_col = col + dc
        new_cell = (new_row, new_col)

        if not is_inside(new_row, new_col):
            info[action] = "WALL"
        elif new_cell in env["obstacles"]:
            info[action] = "OBSTACLE"
        else:
            info[action] = "FREE"

    return info


def get_legal_actions(env):
    info = get_neighbor_info(env)
    legal = []

    for action in info:
        if info[action] == "FREE":
            legal.append(action)

    return legal


def random_environment(dirt_count=4, obstacle_count=2):
    all_cells = [(r, c) for r in range(ROWS) for c in range(COLS)]
    agent_pos = (0, 0)

    while True:
        possible_obstacles = [cell for cell in all_cells if cell != agent_pos]
        obstacle_cells = set(random.sample(possible_obstacles, obstacle_count))

        possible_dirt = [cell for cell in all_cells if cell not in obstacle_cells]
        dirt_cells = set(random.sample(possible_dirt, dirt_count))

        env = create_environment(agent_pos, dirt_cells, obstacle_cells)

        if len(get_legal_actions(env)) > 0:
            return env


def copy_environment(env):
    return {
        "agent_pos": env["agent_pos"],
        "dirt": set(env["dirt"]),
        "obstacles": set(env["obstacles"]),
        "initial_dirt_count": env["initial_dirt_count"],
        "step": env["step"],
        "last_action": env["last_action"]
    }


def is_dirty(env):
    return env["agent_pos"] in env["dirt"]


def sensor(env):
    return {
        "position": env["agent_pos"],
        "dirty_here": is_dirty(env),
        "dirt_left": len(env["dirt"]),
        "legal_actions": get_legal_actions(env),
        "neighbor_info": get_neighbor_info(env),
        "last_action": env["last_action"],
        "step": env["step"]
    }


def performance(env):
    return env["initial_dirt_count"] - len(env["dirt"])


def performance_score(env):
    return performance(env) * 10 - env["step"]


class AgentModel:
    def __init__(self):
        self.current_pos = None
        self.visited = set()
        self.known_clean = set()
        self.known_dirty = set()
        self.known_obstacles = set()
        self.blocked_actions = []
        self.action_history = []

    def update_model(self, percept):
        pos = percept["position"]
        self.current_pos = pos
        self.visited.add(pos)

        if percept["dirty_here"]:
            self.known_dirty.add(pos)
            self.known_clean.discard(pos)
        else:
            self.known_clean.add(pos)
            self.known_dirty.discard(pos)

        row, col = pos

        for action, status in percept["neighbor_info"].items():
            dr, dc = ACTIONS[action]
            near_cell = (row + dr, col + dc)

            if status == "OBSTACLE":
                self.known_obstacles.add(near_cell)

    def remember_action(self, action, rule):
        self.action_history.append((action, rule))


class ModelBasedReflexAgent:
    def __init__(self):
        self.model = AgentModel()

    def decide_action(self, percept):
        self.model.update_model(percept)

        pos = percept["position"]
        legal = percept["legal_actions"]

        if percept["dirty_here"]:
            action = "SUCK"
            rule = "IF ô hiện tại bẩn THEN SUCK"
            self.model.remember_action(action, rule)
            return action, rule

        if percept["dirt_left"] == 0:
            action = "STOP"
            rule = "IF không còn bụi THEN STOP"
            self.model.remember_action(action, rule)
            return action, rule

        next_action = self.get_scan_action(pos)

        if next_action in legal:
            action = next_action
            rule = "IF ô hiện tại sạch THEN đi theo đường quét cố định"
            self.model.remember_action(action, rule)
            return action, rule

        if next_action is not None and next_action not in legal:
            self.model.blocked_actions.append((pos, next_action))

        for action in legal:
            new_pos = self.get_new_position(pos, action)
            if new_pos not in self.model.visited:
                rule = "IF hướng quét bị chặn THEN chọn hướng hợp lệ tới ô chưa đi qua"
                self.model.remember_action(action, rule)
                return action, rule

        if len(legal) > 0:
            action = legal[0]
            rule = "IF không còn ô mới xung quanh THEN chọn hướng hợp lệ đầu tiên"
            self.model.remember_action(action, rule)
            return action, rule

        action = "STOP"
        rule = "IF không có action hợp lệ THEN STOP"
        self.model.remember_action(action, rule)
        return action, rule

    def get_scan_action(self, pos):
        if pos not in SCAN_PATH:
            return None

        index = SCAN_PATH.index(pos)

        if index == len(SCAN_PATH) - 1:
            return None

        next_pos = SCAN_PATH[index + 1]
        return self.action_from_to(pos, next_pos)

    def action_from_to(self, old_pos, new_pos):
        r1, c1 = old_pos
        r2, c2 = new_pos

        if r2 == r1 - 1 and c2 == c1:
            return "UP"
        if r2 == r1 + 1 and c2 == c1:
            return "DOWN"
        if r2 == r1 and c2 == c1 - 1:
            return "LEFT"
        if r2 == r1 and c2 == c1 + 1:
            return "RIGHT"

        return None

    def get_new_position(self, pos, action):
        row, col = pos
        dr, dc = ACTIONS[action]
        return row + dr, col + dc


def apply_action(env, action):
    row, col = env["agent_pos"]

    if action == "SUCK":
        if (row, col) in env["dirt"]:
            env["dirt"].remove((row, col))

    elif action in ACTIONS:
        dr, dc = ACTIONS[action]
        new_row = row + dr
        new_col = col + dc
        new_cell = (new_row, new_col)

        if is_inside(new_row, new_col) and new_cell not in env["obstacles"]:
            env["agent_pos"] = new_cell

    env["step"] += 1
    env["last_action"] = action


def line():
    print("=" * 88)


def print_peas():
    line()
    print("PEAS CHO MÁY HÚT BỤI - MODEL BASED REFLEX AGENT")
    line()

    for key, value in PEAS.items():
        print(key + ":")
        print("  -", value)
        print()


def print_scan_path():
    line()
    print("ĐƯỜNG QUÉT CỐ ĐỊNH CỦA AGENT")
    line()
    print("(0,0) -> (0,1) -> (0,2)")
    print("                    ↓")
    print("(1,0) <- (1,1) <- (1,2)")
    print("↓")
    print("(2,0) -> (2,1) -> (2,2)")
    print()


def cell_symbol(env, cell):
    if cell == env["agent_pos"] and cell in env["dirt"]:
        return "AD"
    if cell == env["agent_pos"]:
        return "A"
    if cell in env["obstacles"]:
        return "X"
    if cell in env["dirt"]:
        return "D"
    return "."


def print_board(env):
    print("+------+------+------+")
    for r in range(ROWS):
        row_text = "|"
        for c in range(COLS):
            symbol = cell_symbol(env, (r, c))
            row_text += f"  {symbol:^2}  |"
        print(row_text)
        print("+------+------+------+")

    print("A = Agent | D = Dirt | X = Vật cản | AD = Agent đứng trên bụi | . = Sạch")
    print()


def print_initial_state(initial_env):
    line()
    print("TRẠNG THÁI BAN ĐẦU")
    line()
    print("Agent position  :", initial_env["agent_pos"])
    print("Dirt cells      :", initial_env["dirt"])
    print("Obstacle cells  :", initial_env["obstacles"])
    print("Số bụi ban đầu  :", initial_env["initial_dirt_count"])
    print("Step ban đầu    :", initial_env["step"])
    print()
    print_board(initial_env)


def print_current_state(env, percept):
    line()
    print("TRẠNG THÁI HIỆN TẠI - SENSOR/PERCEPT")
    line()
    print("Step hiện tại   :", env["step"])
    print("Agent position  :", percept["position"])
    print("Dirty here      :", percept["dirty_here"])
    print("Dirt left       :", percept["dirt_left"])
    print("Legal actions   :", percept["legal_actions"])
    print("Neighbor info   :", percept["neighbor_info"])
    print("Last action     :", percept["last_action"])
    print("Performance     :", f"{performance(env)} / {env['initial_dirt_count']}")
    print("Score           :", performance_score(env))
    print()
    print_board(env)


def print_agent_decision(action, rule):
    line()
    print("MODEL BASED REFLEX AGENT DECISION")
    line()
    print("Rule   :", rule)
    print("Action :", action)
    print()


def print_history(history):
    line()
    print("LỊCH SỬ ACTION")
    line()

    if len(history) == 0:
        print("Chưa có action nào.")
    else:
        for item in history:
            print(item)

    print()


def print_result(env):
    line()
    print("KẾT QUẢ")
    line()

    if len(env["dirt"]) == 0:
        print("Trạng thái: ĐÃ HÚT SẠCH BỤI")
    else:
        print("Trạng thái: CHƯA HÚT SẠCH BỤI")

    print("Tổng số bước      :", env["step"])
    print("Bụi đã hút        :", f"{performance(env)} / {env['initial_dirt_count']}")
    print("Score cuối        :", performance_score(env))
    line()


def run_model_based_agent(env, max_steps=50, delay=0.5, animate=True):
    agent = ModelBasedReflexAgent()
    initial_env = copy_environment(env)
    history = []

    while env["step"] < max_steps:
        if animate:
            clear_output(wait=True)

        percept = sensor(env)
        action, rule = agent.decide_action(percept)

        print_peas()
        print_scan_path()
        print_initial_state(initial_env)
        print_current_state(env, percept)
        print_agent_decision(action, rule)
        print_history(history)

        if action == "STOP":
            print_result(env)
            break

        old_pos = env["agent_pos"]
        old_dirty = percept["dirty_here"]
        old_perf = performance(env)
        old_score = performance_score(env)

        apply_action(env, action)

        history.append(
            f"Step {env['step']:02d} | Action: {action:<5} | "
            f"Position: {old_pos} -> {env['agent_pos']} | "
            f"Dirty before: {old_dirty} | "
            f"Performance: {old_perf}/{env['initial_dirt_count']} -> {performance(env)}/{env['initial_dirt_count']} | "
            f"Score: {old_score} -> {performance_score(env)}"
        )

        if len(env["dirt"]) == 0:
            if animate:
                clear_output(wait=True)

            percept = sensor(env)
            print_peas()
            print_scan_path()
            print_initial_state(initial_env)
            print_current_state(env, percept)
            print_history(history)
            print_result(env)
            break

        time.sleep(delay)

    if env["step"] >= max_steps and len(env["dirt"]) > 0:
        print_result(env)


def run_manual(env):
    initial_env = copy_environment(env)
    history = []

    while True:
        clear_output(wait=True)
        percept = sensor(env)

        print_peas()
        print_scan_path()
        print_initial_state(initial_env)
        print_current_state(env, percept)
        print_history(history)

        if percept["dirt_left"] == 0:
            print_result(env)
            break

        print("Nhập action: SUCK / UP / DOWN / LEFT / RIGHT / STOP")
        action = input("Action = ").upper().strip()

        if action == "STOP":
            print("Bạn đã dừng manual mode.")
            print_result(env)
            break

        if action not in ["SUCK", "UP", "DOWN", "LEFT", "RIGHT"]:
            history.append(f"Action '{action}' không tồn tại.")
            continue

        if action in ACTIONS and action not in percept["legal_actions"]:
            history.append(f"Action '{action}' không hợp lệ vì ra ngoài phòng hoặc đụng vật cản.")
            continue

        old_pos = env["agent_pos"]
        old_dirty = percept["dirty_here"]
        old_perf = performance(env)
        old_score = performance_score(env)

        apply_action(env, action)

        history.append(
            f"Step {env['step']:02d} | Action: {action:<5} | "
            f"Position: {old_pos} -> {env['agent_pos']} | "
            f"Dirty before: {old_dirty} | "
            f"Performance: {old_perf}/{env['initial_dirt_count']} -> {performance(env)}/{env['initial_dirt_count']} | "
            f"Score: {old_score} -> {performance_score(env)}"
        )


def main():
    clear_output(wait=True)

    print("CHƯƠNG TRÌNH MÁY HÚT BỤI - MODEL BASED REFLEX AGENT")
    print("Chạy trên Jupyter Notebook / Google Colab")
    print()

    try:
        dirt_count = int(input("Nhập số ô bụi random từ 1 đến 8: "))
    except:
        dirt_count = 4

    try:
        obstacle_count = int(input("Nhập số vật cản random từ 0 đến 4: "))
    except:
        obstacle_count = 2

    if obstacle_count < 0:
        obstacle_count = 0
    if obstacle_count > 4:
        obstacle_count = 4

    max_dirt = ROWS * COLS - obstacle_count
    if dirt_count < 1:
        dirt_count = 1
    if dirt_count > max_dirt:
        dirt_count = max_dirt

    env = random_environment(dirt_count, obstacle_count)

    print()
    print("CHỌN CHẾ ĐỘ")
    print("1. MANUAL - Người chơi tự nhập action")
    print("2. AGENT - Model Based Reflex Agent tự chạy")
    choice = input("Nhập 1 hoặc 2: ").strip()

    if choice == "1":
        run_manual(env)
    elif choice == "2":
        try:
            delay = float(input("Nhập delay giữa các bước, ví dụ 0.5: "))
        except:
            delay = 0.5

        run_model_based_agent(env, max_steps=50, delay=delay, animate=True)
    else:
        print("Bạn nhập sai chế độ.")


main()


PEAS CHO MÁY HÚT BỤI - MODEL BASED REFLEX AGENT
P - Performance:
  - Hút sạch bụi, tránh vật cản, số bước càng ít càng tốt

E - Environment:
  - Phòng 3x3, có ô sạch, ô bẩn và có vật cản

A - Actuators:
  - SUCK, UP, DOWN, LEFT, RIGHT, STOP

S - Sensors:
  - Biết vị trí hiện tại, biết ô hiện tại có bụi không, biết action hợp lệ, biết vật cản xung quanh

ĐƯỜNG QUÉT CỐ ĐỊNH CỦA AGENT
(0,0) -> (0,1) -> (0,2)
                    ↓
(1,0) <- (1,1) <- (1,2)
↓
(2,0) -> (2,1) -> (2,2)

TRẠNG THÁI BAN ĐẦU
Agent position  : (0, 0)
Dirt cells      : {(0, 2), (2, 2), (0, 0), (1, 0), (2, 0)}
Obstacle cells  : {(0, 1), (2, 1)}
Số bụi ban đầu  : 5
Step ban đầu    : 0

+------+------+------+
|  AD  |  X   |  D   |
+------+------+------+
|  D   |  .   |  .   |
+------+------+------+
|  D   |  X   |  D   |
+------+------+------+
A = Agent | D = Dirt | X = Vật cản | AD = Agent đứng trên bụi | . = Sạch

TRẠNG THÁI HIỆN TẠI - SENSOR/PERCEPT
Step hiện tại   : 49
Agent position  : (2, 0)
Dirty here      : Fal